In [1]:
import cryo
import polars as pl
import binascii
import web3
import json
from eth_abi import decode
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime
import pandas as pd

In [2]:
# The multical cantract address, but we also need ABI
MULTICALL3_ADDRESS = '0xcA11bde05977b3631167028862bE2a173976CA11'
MULTICALL3_ABI=json.loads('[{"inputs":[{"internalType":"bool","name":"requireSuccess","type":"bool"},{"components":[{"internalType":"address","name":"target","type":"address"},{"internalType":"bytes","name":"callData","type":"bytes"}],"internalType":"struct Multicall3.Call[]","name":"calls","type":"tuple[]"}],"name":"tryAggregate","outputs":[{"components":[{"internalType":"bool","name":"success","type":"bool"},{"internalType":"bytes","name":"returnData","type":"bytes"}],"internalType":"struct Multicall3.Result[]","name":"returnData","type":"tuple[]"}],"stateMutability":"payable","type":"function"}]')

In [3]:
# Contract Addresses
UNIV3_USDC_ETH='0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640'

In [4]:
# Function Signatures 4 bytes
getBlocknumber_4b = '42cbb15c'
getBloclTimestamp_4b= '0f28c97d'
getReserves_4b = '0902f1ac'
slot0_4b = '3850c7bd'

In [5]:
# Functions
def bytes_to_hexstr(b: any) -> str:
    if isinstance(b,list):
        return [bytes_to_hexstr(a) for a in b]
    return '0x' + b.hex()

def decode_outputdata_uniV3_price(b: bytes) -> list[float]:
    aggregated_data_uniV3 = decode(['(bool,bytes)[]'], b)[0]

    # UNI-V3
    # slot0():
    # sqrtPriceX96 uint160, tick int24, observationIndex uint16, observationCardinality uint16, observationCardinalityNext uint16, feeProtocol uint8, unlocked bool
    # 'uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'
    usdc_eth_slot0_raw = aggregated_data_uniV3[0]
    usdc_eth_slot0_sqrt_ratioX96 = decode(['uint160', 'int24', 'uint16', 'uint16', 'uint16', 'uint8', 'bool'], usdc_eth_slot0_raw[1])[0]
    usdc_eth_price = usdc_eth_slot0_sqrt_ratioX96**2 / 2**192 /1e12
    eth_usdc_price_v3 = 1/usdc_eth_price

    # Timestamp
    timestamp_raw = aggregated_data_uniV3[-1]
    timestamp = int(timestamp_raw[1].hex(),16)
    
    return [eth_usdc_price_v3, timestamp]

In [6]:
# web3 instance, function from web3py
w3 = web3.Web3()
m3 = w3.eth.contract(address = MULTICALL3_ADDRESS, abi=MULTICALL3_ABI)

In [7]:
# Arguments fro the tryAggregate Fuunction
aggregate_calldata = [
    [
        UNIV3_USDC_ETH,
        f'0x{slot0_4b}',
    ],
    [
        MULTICALL3_ADDRESS,
        f'0x{getBloclTimestamp_4b}'
    ],
]

In [8]:
# aggregate_calldata list of list
# Generate calldate (the input) via m3 Multicall3 encode ABIfor cryo - calldata is in Hex format
calldata = m3.encode_abi("tryAggregate", args=[False, aggregate_calldata])

In [9]:
# cryo.collect() using calldata 
cryo_kwargs = {
    'rpc': 'https://eth.merkle.io',
    'blocks': ['-100:latest'], 
}
            
eth_call_uni_df = cryo.collect(
    'eth_calls',
    to_address = [MULTICALL3_ADDRESS],
    call_data=[calldata],
     output_format="polars",
    **cryo_kwargs,
)

In [10]:
# Now Output in Binary Format
output_data=eth_call_uni_df['output_data'][0]

In [11]:
# Function decode_outputdata_uniV3_price uses eth_abi.decode to get decimals values from binary format
prices = [decode_outputdata_uniV3_price(x) for x in eth_call_uni_df['output_data'].to_list()]

TypeError: The `data` value must be of bytes type. Got <class 'NoneType'>

In [27]:
df_prices_row = pd.DataFrame(prices, columns=['eth/usdc uniV3','timestamp'])
df_prices_row['Date'] = pd.to_datetime(df_prices_row['timestamp'], unit='s')

In [28]:
df_prices = df_prices_row.copy()

In [29]:
df_prices = df_prices.set_index('Date').drop('timestamp', axis=1)

In [30]:
df_prices.head(20)

,eth/usdc uniV3
Date,
2025-05-19 08:26:47,2414.527723
2025-05-19 08:26:59,2414.526582
2025-05-19 08:27:11,2414.526582
2025-05-19 08:27:23,2413.196332
2025-05-19 08:27:35,2412.967595
2025-05-19 08:27:47,2412.468844
2025-05-19 08:27:59,2412.463138
2025-05-19 08:28:11,2412.449202
2025-05-19 08:28:23,2412.443511
